In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg, sum as spark_sum, round as spark_round

print("Initializing Spark Session...")

spark = SparkSession.builder \
    .appName("Steam_Batch_Processing") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.4") \
    .getOrCreate()

print(f"Spark version: {spark.version} is READY!")

Initializing Spark Session...
Spark version: 3.5.0 is READY!


In [2]:
print("Loading data from Bronze Layer...")

df_recs = spark.read.csv("data/raw/recommendations.csv", header=True, inferSchema=True)

df_games = spark.read.csv("data/raw/games.csv", header=True, inferSchema=True)

print(f"Total Recommendations loaded: {df_recs.count():,}")

Loading data from Bronze Layer...
Total Recommendations loaded: 41,154,794


In [4]:
print("Cleaning and Joining data...")

df_recs_clean = df_recs.dropna(subset=["app_id", "hours", "is_recommended"])

df_joined = df_recs_clean.join(df_games, on="app_id", how="inner")

df_transformed = df_joined.withColumn(
    "is_positive", 
    when(col("is_recommended") == True, 1).otherwise(0)
)

Cleaning and Joining data...


In [5]:
print("Aggregating insights...")

df_insights = df_transformed.groupBy("app_id", "title", "price_final") \
    .agg(
        count("review_id").alias("total_reviews"),
        spark_sum("is_positive").alias("positive_recommendations"),
        spark_round(avg("hours"), 2).alias("avg_hours_played")
    ) \
    .filter(col("total_reviews") >= 500)

df_final = df_insights.withColumn(
    "recommendation_rate",
    spark_round((col("positive_recommendations") / col("total_reviews")) * 100, 2)
)

df_final.orderBy(col("total_reviews").desc()).show(5)

Aggregating insights...
+-------+--------------------+-----------+-------------+------------------------+----------------+-------------------+
| app_id|               title|price_final|total_reviews|positive_recommendations|avg_hours_played|recommendation_rate|
+-------+--------------------+-----------+-------------+------------------------+----------------+-------------------+
|    440|     Team Fortress 2|        0.0|       319492|                  294879|          318.66|               92.3|
| 252490|                Rust|       40.0|       270684|                  226293|          343.37|               83.6|
|1091500|      Cyberpunk 2077|       60.0|       226414|                  168631|          102.23|              74.48|
|    730|Counter-Strike: G...|       15.0|       219737|                  186306|          428.97|              84.79|
|    570|              Dota 2|        0.0|       216914|                  171841|          429.08|              79.22|
+-------+---------------

In [6]:
print("Saving to Storage Layers...")

parquet_path = "data/processed/steam_transformed.parquet"
df_transformed.write.mode("overwrite").parquet(parquet_path)
print(f"[SUCCESS] Transformed data saved to Silver Layer at {parquet_path}")

db_properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}
db_url = "jdbc:postgresql://postgres-db:5432/steam_analytics"

df_final.write.jdbc(
    url=db_url,
    table="game_insights",
    mode="overwrite",
    properties=db_properties
)
print("[SUCCESS] Aggregated insights saved to Gold Layer (PostgreSQL)!")

Saving to Storage Layers...
[SUCCESS] Transformed data saved to Silver Layer at data/processed/steam_transformed.parquet
[SUCCESS] Aggregated insights saved to Gold Layer (PostgreSQL)!
